In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


# 1. Ingestão de Dados
df_cancer = pd.read_csv('wisconsin.csv')

# 2. Saneamento (Limpando o ruído sistêmico)
if 'id' in df_cancer.columns:
    df_cancer = df_cancer.drop(columns=['id'])
if 'Unnamed: 32' in df_cancer.columns:
    df_cancer = df_cancer.drop(columns=['Unnamed: 32'])

# Transformar M/B em 1/0
le = LabelEncoder()
df_cancer['diagnosis'] = le.fit_transform(df_cancer['diagnosis'])

# 3. Divisão Treino/Teste
X_c = df_cancer.drop(columns=['diagnosis'])
y_c = df_cancer['diagnosis']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# 4. Padronização
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

# 5. Treinamento do Motor
modelo_cancer = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_cancer.fit(X_train_c_scaled, y_train_c)

# 6. Relatório de Execução
previsoes_cancer = modelo_cancer.predict(X_test_c_scaled)
print("--- RELATÓRIO DO MOTOR: CÂNCER DE MAMA ---")
print(f"Acurácia: {accuracy_score(y_test_c, previsoes_cancer):.2%}")
print(classification_report(y_test_c, previsoes_cancer, target_names=le.classes_))

--- RELATÓRIO DO MOTOR: CÂNCER DE MAMA ---
Acurácia: 96.49%
              precision    recall  f1-score   support

           B       0.96      0.99      0.97        71
           M       0.98      0.93      0.95        43

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Ingestão e Limpeza Básica
df_vida = pd.read_csv('expectancy.csv')
df_vida.columns = df_vida.columns.str.strip()
df_vida = df_vida.dropna(subset=['Life expectancy'])

X_v = df_vida.drop(columns=['Life expectancy', 'Country'])
y_v = df_vida['Life expectancy']

# 2. Arquitetura do Tratamento
cols_num = X_v.select_dtypes(include=['int64', 'float64']).columns
cols_cat = X_v.select_dtypes(include=['object']).columns

tratamento_num = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
tratamento_cat = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', tratamento_num, cols_num),
    ('cat', tratamento_cat, cols_cat)
])

# 3. Treino/Teste e Modelagem
X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_v, y_v, test_size=0.2, random_state=42)

pipeline_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])
pipeline_reg.fit(X_train_v, y_train_v)

# 4. Relatório de Execução
previsoes_vida = pipeline_reg.predict(X_test_v)
print("\n--- RELATÓRIO DO MOTOR: EXPECTATIVA DE VIDA ---")
print(f"Erro Médio Absoluto (MAE): {mean_absolute_error(y_test_v, previsoes_vida):.2f} anos")
print(f"Capacidade Explicativa (R²): {r2_score(y_test_v, previsoes_vida):.2%}")

/tmp/ipykernel_11096/3879087512.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_cat = X_v.select_dtypes(include=['object']).columns



--- RELATÓRIO DO MOTOR: EXPECTATIVA DE VIDA ---
Erro Médio Absoluto (MAE): 1.05 anos
Capacidade Explicativa (R²): 96.78%


In [3]:
import xgboost as xgb

# 1. Instanciar o XGBoost
modelo_xgb = xgb.XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=4, 
    random_state=42,
    eval_metric='logloss'
)

# 2. Treinar
modelo_xgb.fit(X_train_c_scaled, y_train_c)

# 3. Prever e Avaliar
previsoes_xgb = modelo_xgb.predict(X_test_c_scaled)
print("\n--- RELATÓRIO DO MOTOR: XGBOOST (CÂNCER) ---")
print(f"Acurácia XGBoost: {accuracy_score(y_test_c, previsoes_xgb):.2%}")


--- RELATÓRIO DO MOTOR: XGBOOST (CÂNCER) ---
Acurácia XGBoost: 95.61%
